# attenzione: fare esercizzi lezione 6

# discesa gradiente stocastica
**discesa gradiente**:
* per calcolare $\nabla J(w)$ usa l'intero dataset, ossia la matrice $X$
* se $X$ enorme, non e' possibile usare la discesa del gradiente.

**discesa gradiente stocastica**: divido $X$ in blocchi, ciascuno con un numero fissato di righe, possibilmente ogni blocco le sceglie randomicamente da $X$.
* $X_b$ e' la matrice $b$ esima, ottenuta.
* **a che serve**: non calcolo il gradiente su $X$, ma piuttosto su un unico $X_i$
* **stocastica**: voglio che ogni blocco abbia righe scelte in modo uniforme a random, in modo che un blocco sia rappresentativo dell'intero data set (gemini conferma)

$$
\nabla J(w) = - X^T_b \times \text{errors}
$$

**pro dell'approccio stocastico**:
* moltiplicazioni con matrici piccole
* vedremo che converge prima


In [ ]:
import matplotlib.pyplot as pyplot
import numpy as np

class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size
    
    def fit(self, X, y):
        self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_

        # Standardizzazione delle feature
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_
        
        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])

        for _ in range(self.n_iter):
            # crea e permuta gli indici
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            # itera a step di batch_size
            for b_idx in range(0, X.shape[0], self.batch_size_):
                # prendi i valori corrispondenti al blocco
                idxs_b = shuffle_idxs[b_idx:b_idx+self.batch_size_]
                X_b = X_std[idxs_b]
                y_b = y[idxs_b]
                
                # calcola la funzione in X_b e calcola gli errori9
                output = self.net_input(X_b)
                errors = y_b - output

                # aggiorna i pesi
                self.w_[1:] += self.eta * X_b.T.dot(errors)/self.batch_size_
                self.w_[0] += self.eta * errors.sum()/self.batch_size_
            
            # ora fai la predizione su tutte le righe e vedi quant'e' l'errore
            output = self.net_input(X_std)
            self.costi_.append(mae(y,output))
            if len(self.costi_) >= 2:
                if np.abs(self.costi_[-1]-self.costi_[-2]) < self.tol_: break

    # il resto non cambia!
    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef

def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

# sperimentiamo con `sklearn`

In [ ]:
from sklearn.datasets import make_regression